# 02_clean_orders_day01 — Bronze → Silver (local)

**Week 2 · Pipeline Day 1** — Your first **mini ETL**: read messy order rows in Python, apply rules, print a small metrics report. **No AWS, no pandas** — only lists, dicts, and loops (from `01_structures_demo`).

**How to use this notebook**
- Run cells **top to bottom** (`Shift + Enter`).
- Read the rules in **§1** before coding or running the solution cells.
- Matching script (same logic): `Week_2/Day_01/scripts/02_clean_orders_day01.py`

---
## §0. Scenario

You receive **six raw JSON-like rows** (`RAW_ORDERS`) from a bronze landing zone. Data quality issues:

- Some orders are **not** `completed` (`pending`, `cancelled`).
- One **completed** row has **`amount` = `None`** (NULL).
- **City** strings have extra spaces and mixed case (`" hyd "`, `" Chennai "`).

Build a **`cleaned`** list of rows that are safe for downstream analytics, then print pipeline KPIs.

---
## §1. Your task (acceptance criteria)

Implement a loop over `RAW_ORDERS` that does **all** of the following.

### A. Filter (drop bad rows)

| Rule | Python idea | SQL mental model |
|------|-------------|------------------|
| Keep only finished sales | `status == "completed"` | `WHERE status = 'completed'` |
| Drop NULL amounts | `amount is not None` | `WHERE amount IS NOT NULL` |

Rows **removed** by these rules: `order_id` **2** (pending), **3** (NULL amount), **5** (cancelled).

### B. Transform (normalize surviving rows)

For each row you keep, copy the dict and set:

- **`city`** → `row["city"].strip().upper()` (trim spaces, uppercase — e.g. `" hyd "` → `"HYD"`)

Use `{**row, "city": ...}` or equivalent so other columns stay unchanged.

### C. Aggregates & report (after the loop)

1. **`unique_users`** — `set` of `user_id` from `cleaned` (distinct customers).
2. **`total_revenue`** — `sum` of `amount` on `cleaned`.
3. **Print** one summary line, sorted users, then each cleaned row.

### When your logic is correct, you should see:

```text
Pipeline Day 1 | valid_orders=3 | unique_users=3 | revenue=485.50
Users seen: ['u1', 'u2', 'u3']
```

and **three** dicts: `order_id` **1**, **4**, and **6** with cities **HYD**, **BLR**, **DELHI**.

---
## §2. Input data (do not edit)

Run the cell below to load the bronze batch into memory.

In [ ]:
from __future__ import annotations

RAW_ORDERS = [
    {"order_id": 1, "user_id": "u1", "status": "completed", "amount": 120.0, "city": " hyd "},
    {"order_id": 2, "user_id": "u2", "status": "pending", "amount": 90.0, "city": "blr"},
    {"order_id": 3, "user_id": "u1", "status": "completed", "amount": None, "city": " Chennai "},
    {"order_id": 4, "user_id": "u3", "status": "completed", "amount": 55.5, "city": "blr"},
    {"order_id": 5, "user_id": "u4", "status": "cancelled", "amount": 200.0, "city": "hyd"},
    {"order_id": 6, "user_id": "u2", "status": "completed", "amount": 310.0, "city": " delhi "},
]

print(f"bronze rows loaded: {len(RAW_ORDERS)}")
for row in RAW_ORDERS:
    print(row)

---
## §3. Solution — filter + normalize loop

**Instructor / self-check path:** Run this cell (or type the same logic yourself in a new cell above before peeking).

Steps inside the loop:
1. `continue` if `status` is not `"completed"`.
2. `continue` if `amount` is `None`.
3. Append a normalized copy with cleaned `city`.

In [ ]:
cleaned = []
for row in RAW_ORDERS:
    if row.get("status") != "completed":
        continue
    if row.get("amount") is None:
        continue
    normalized = {
        **row,
        "city": row["city"].strip().upper(),
    }
    cleaned.append(normalized)

print(f"after filter+normalize → {len(cleaned)} silver rows")
for row in cleaned:
    print(row)

---
## §4. Solution — metrics & report

In [ ]:
unique_users = {r["user_id"] for r in cleaned}
total_revenue = sum(r["amount"] for r in cleaned)

print(
    f"Pipeline Day 1 | valid_orders={len(cleaned)} "
    f"| unique_users={len(unique_users)} "
    f"| revenue={total_revenue:.2f}"
)
print(f"Users seen: {sorted(unique_users)}")